# TEMPLATE 업서트 노트북

이 노트북은 `modular.db`의 `TEMPLATE` 테이블에 데이터를 직접 업서트하기 위한 템플릿입니다.

실행 전 확인:
1. `ITEM.template_id`가 참조할 템플릿 ID를 먼저 정합니다.
2. `type`은 렌더링 유형을 구분하는 문자열로 작성합니다.
3. 이후 `ITEM`, `CHOICE` 업서트에서 동일한 `template_id`를 사용합니다.


## TEMPLATE 테이블 컬럼

- `id`: 템플릿 고유 ID
- `type`: 템플릿 유형
- `description`: 템플릿 설명(nullable)


In [5]:
# 1) 테이블 이름 + 스키마 설정
from pathlib import Path
import re
import sqlite3

DB_PATH = Path(r"C:\Users\user\workspace\2.0-modular\docs\modular.db")
TABLE_NAME = 'TEMPLATE'

SCHEMA = [
    {'name': 'id', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'type', 'type': 'TEXT', 'constraints': 'NOT NULL'},
    {'name': 'description', 'type': 'TEXT', 'constraints': ''},
]
PRIMARY_KEY = ['id']

VALID_TYPES = {'INTEGER', 'REAL', 'TEXT', 'BLOB'}
identifier_re = re.compile(r'^[A-Za-z_][A-Za-z0-9_\.]*$')

def validate_identifier(name: str) -> None:
    if not identifier_re.match(name):
        raise ValueError(f'잘못된 이름: {name} (영문/숫자/언더스코어/점만 허용, 숫자로 시작 불가)')

validate_identifier(TABLE_NAME)
seen = set()
for col in SCHEMA:
    col_name = col['name']
    col_type = col['type'].upper()
    validate_identifier(col_name)
    if col_name in seen:
        raise ValueError(f'중복 컬럼명: {col_name}')
    if col_type not in VALID_TYPES:
        raise ValueError(f'지원하지 않는 타입: {col_type}')
    seen.add(col_name)

print('설정 확인 완료')
print('DB_PATH =', DB_PATH)
print('TABLE_NAME =', TABLE_NAME)
print('SCHEMA =', SCHEMA)
print('PRIMARY_KEY =', PRIMARY_KEY)


설정 확인 완료
DB_PATH = C:\Users\user\workspace\2.0-modular\docs\modular.db
TABLE_NAME = TEMPLATE
SCHEMA = [{'name': 'id', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'type', 'type': 'TEXT', 'constraints': 'NOT NULL'}, {'name': 'description', 'type': 'TEXT', 'constraints': ''}]
PRIMARY_KEY = ['id']


In [6]:
# 2) TEMPLATE 테이블 구조 확인
conn = sqlite3.connect(DB_PATH)
try:
    cur = conn.execute('PRAGMA table_info("TEMPLATE")')
    print('table_info(TEMPLATE)')
    for row in cur.fetchall():
        print(row)
finally:
    conn.close()


table_info(TEMPLATE)
(0, 'id', 'TEXT', 1, None, 1)
(1, 'type', 'TEXT', 1, None, 0)
(2, 'description', 'TEXT', 0, None, 0)


## 업서트


In [7]:
# 3) TEMPLATE 테이블 업서트
import json

insert_columns = []
schema_type_map = {}
for col in SCHEMA:
    c_name = col['name']
    c_type = col['type'].upper()
    c_constraints = col.get('constraints', '').upper()
    schema_type_map[c_name] = c_type
    if 'PRIMARY KEY' in c_constraints and 'AUTOINCREMENT' in c_constraints:
        continue
    insert_columns.append(c_name)

print('입력 대상 컬럼:', insert_columns)

ROWS = [
    {
        'id': 'TPL_LIKERT',
        'type': 'likert',
        'description': '리커트 척도',
    },
    {
        'id': 'TPL_BIPOLAR_WITH_PROMPT',
        'type': 'bipolar_with_prompt',
        'description': '양극형 선택지 + 문항 프롬프트',
    },
    {
        'id': 'TPL_LIKERT_MATRIX',
        'type': 'likert_matrix',
        'description': '행렬형 리커트 응답',
    },
    {
        'id': 'TPL_BIPOLAR_LABELS_ONLY',
        'type': 'bipolar_labels_only',
        'description': '양 끝 라벨만 노출되는 양극형 선택지',
    },
]

def normalize_value(col_name: str, value):
    col_type = schema_type_map[col_name]
    if col_type == 'TEXT' and isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return value

expected_keys = set(insert_columns)
normalized_rows = []
for i, row in enumerate(ROWS, start=1):
    if set(row.keys()) != expected_keys:
        raise ValueError(
            f'{i}번째 ROW 키 불일치: expected={sorted(expected_keys)}, got={sorted(row.keys())}. '
            'SCHEMA 컬럼명과 ROW 키 이름을 정확히 맞추세요.'
        )
    normalized = {k: normalize_value(k, row[k]) for k in insert_columns}
    normalized_rows.append(normalized)

placeholders = ', '.join(['?'] * len(insert_columns))
col_sql = ', '.join([f'"{c}"' for c in insert_columns])
conflict_columns = ['id']
update_columns = [c for c in insert_columns if c not in conflict_columns]
conflict_sql = ', '.join([f'"{c}"' for c in conflict_columns])
update_sql = ', '.join([f'"{c}" = excluded."{c}"' for c in update_columns])
insert_sql = (
    f'INSERT INTO "{TABLE_NAME}" ({col_sql}) VALUES ({placeholders}) '
    f'ON CONFLICT ({conflict_sql}) DO UPDATE SET {update_sql}'
)
values = [tuple(row[c] for c in insert_columns) for row in normalized_rows]

print('실행 SQL:', insert_sql)
print('입력 건수:', len(values))

conn = sqlite3.connect(DB_PATH)
try:
    conn.executemany(insert_sql, values)
    conn.commit()
    print(f'업서트 완료: {len(values)}건')
finally:
    conn.close()


입력 대상 컬럼: ['id', 'type', 'description']
실행 SQL: INSERT INTO "TEMPLATE" ("id", "type", "description") VALUES (?, ?, ?) ON CONFLICT ("id") DO UPDATE SET "type" = excluded."type", "description" = excluded."description"
입력 건수: 4
업서트 완료: 4건


In [8]:
# 4) 결과 조회
conn = sqlite3.connect(DB_PATH)
try:
    cur = conn.execute('SELECT * FROM "TEMPLATE" ORDER BY "id"')
    rows = cur.fetchall()
    print(f'총 {len(rows)}건')
    for row in rows:
        print(row)
finally:
    conn.close()


총 4건
('TPL_BIPOLAR_LABELS_ONLY', 'bipolar_labels_only', '양 끝 라벨만 노출되는 양극형 선택지')
('TPL_BIPOLAR_WITH_PROMPT', 'bipolar_with_prompt', '양극형 선택지 + 문항 프롬프트')
('TPL_LIKERT', 'likert', '리커트 척도')
('TPL_LIKERT_MATRIX', 'likert_matrix', '행렬형 리커트 응답')
